# SAM + augmentation consistency pipeline on Colab

Runs the full pipeline (no SAM3D) on 100–150 images:
1. Clone repo (or use your GitHub URL)
2. Install deps + download SAM checkpoint
3. Download MIT Indoor, sample 100–150 images (3 classes)
4. Run SAM on originals → masks + metadata
5. Generate augmented images (even: hflip / rotation / resize / blur)
6. Run SAM on augmented images
7. Evaluate consistency → analysis → plots

**Runtime:** GPU recommended (faster SAM). Results can be zipped and downloaded.

## 1. Clone repo and go to project dir

Push your `sam-household-masks` code to a GitHub repo, then paste the clone URL below (use HTTPS, e.g. `https://github.com/USERNAME/sam-household-masks.git`). If you already have the repo elsewhere, you can skip clone and set `PROJECT_DIR` manually.

In [ ]:
# @param {type:"string"}
REPO_URL = ""  # e.g. https://github.com/YOUR_USER/sam-household-masks.git

import os

if REPO_URL.strip():
    !git clone --depth 1 {REPO_URL} sam-household-masks 2>/dev/null || (cd sam-household-masks && git pull)
    PROJECT_DIR = "/content/sam-household-masks"
else:
    PROJECT_DIR = "/content/sam-household-masks"  # or upload the folder and set this path
    if not os.path.isdir(PROJECT_DIR):
        raise SystemExit("Set REPO_URL or upload sam-household-masks to /content/sam-household-masks")

os.chdir(PROJECT_DIR)
print("Project dir:", os.getcwd())

## 2. Install dependencies and SAM checkpoint

In [ ]:
!pip install -q torch torchvision numpy opencv-python Pillow tqdm scipy matplotlib
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

In [ ]:
# Download SAM checkpoint (ViT-B, ~375 MB)
CHECKPOINT = "sam_vit_b_01ec64.pth"
if not os.path.isfile(os.path.join(PROJECT_DIR, CHECKPOINT)):
    !wget -q -O {CHECKPOINT} https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
    print("Downloaded", CHECKPOINT)
else:
    print("Checkpoint already present:", CHECKPOINT)

## 3. Download MIT Indoor and sample 100–150 images (3 classes)

In [ ]:
# @param {type:"integer"}
TOTAL_IMAGES = 120  # 100–150; will be split evenly across bedroom, kitchen, livingroom

import subprocess
subprocess.run(["python", "download_mit_indoor.py", "--work-dir", "/content", "--output-dir", "images/mit_indoor_subset", "--total", str(TOTAL_IMAGES), "--force"], check=True, cwd=PROJECT_DIR)

import glob
n_imgs = len(glob.glob(os.path.join(PROJECT_DIR, "images/mit_indoor_subset", "*.*")))
print("Images in subset:", n_imgs)

## 4. Run SAM on original images

In [ ]:
!python batch_sam_masks.py --images images/mit_indoor_subset --output sam_output/original --checkpoint sam_vit_b_01ec64.pth --model vit_b

## 5. Generate augmented images (even: hflip, rotation, resize, blur)

In [ ]:
!python generate_augmented.py --images images/mit_indoor_subset --output-dir images/mit_indoor_aug --manifest sam_output/aug_manifest.json --augs-per-image 1

## 6. Run SAM on augmented images

In [ ]:
!python batch_sam_masks.py --images images/mit_indoor_aug --output sam_output/augmented --checkpoint sam_vit_b_01ec64.pth --model vit_b

## 7. Evaluate consistency, analyze, plot

In [ ]:
!python evaluate_consistency.py --original-output sam_output/original --augmented-output sam_output/augmented --manifest sam_output/aug_manifest.json --results sam_output/consistency_results.json

In [ ]:
!python analyze_consistency.py --results sam_output/consistency_results.json --original-metadata sam_output/original/metadata --output sam_output/analysis.json

In [ ]:
!python plot_consistency.py --analysis sam_output/analysis.json --output-dir sam_output/plots

## 8. Zip results and download (optional)

In [ ]:
os.chdir(PROJECT_DIR)
!zip -r sam_pipeline_results.zip sam_output/ images/mit_indoor_subset/ images/mit_indoor_aug/ -x "*.pyc" 2>/dev/null
from google.colab import files
zip_path = os.path.join(PROJECT_DIR, "sam_pipeline_results.zip")
if os.path.isfile(zip_path):
    files.download(zip_path)
    print("Download started: sam_pipeline_results.zip")
else:
    print("Zip not found; download sam_output/ and images/ from the file browser.")